# 04 — Feature Engineering (Phase 2)

Applies `features.py`'s lag/rolling/rate-of-change/dew-point functions
(built in Phase 1 but not yet used for modeling) across four cities, and
produces the final train/val/test tables Phase 3 (`05_model_training.ipynb`)
will consume.

**Cities:** Basel, Oslo, Perpignan, De Bilt — climatically diverse, and
Oslo/De Bilt both report wind, so this data can eventually speak to the
"is wind worth a dedicated sensor" open question from the brief.

**Important design note:** lag/rolling features are computed **per city,
before concatenation** — never across a city boundary. Oslo's first day
has no physical relationship to Basel's last day, so stacking cities
first and then computing rolling windows would leak nonsense into the
first few rows of every city except the first.

In [1]:
import sys
sys.path.insert(0, '../src')

import os
import pandas as pd

from data_loader import load_tier1_raw, get_city_frame
from label_rules import build_labels_for_horizon
from features import build_feature_set

In [2]:
def build_training_table(raw_df, cities, horizon=1, lags=(1, 2, 3), rolling_windows=(3, 7)):
    """
    Label + feature-engineer each city SEPARATELY, then concatenate.
    Cities missing 'temp_min' (e.g. Budapest) are skipped, not silently
    dropped -- you'll see a printed warning if that ever happens here.
    """
    tables = []
    for city in cities:
        city_df = get_city_frame(raw_df, city)
        if 'temp_min' not in city_df.columns:
            print(f"Skipping {city}: no temp_min column.")
            continue
        labeled = build_labels_for_horizon(city_df, horizon=horizon)
        featured = build_feature_set(
            labeled, temp_col='temp_min', rh_percent_col='humidity_pct',
            lag_columns=['temp_min', 'humidity_pct'], lags=lags,
            rolling_columns=['temp_min', 'humidity_pct'], rolling_windows=rolling_windows,
        )
        tables.append(featured)
    return pd.concat(tables, ignore_index=True)

In [3]:
CITIES = ['BASEL', 'OSLO', 'PERPIGNAN', 'DE_BILT']

raw = load_tier1_raw('../data/raw/weather_prediction_dataset.csv')
combined = build_training_table(raw, CITIES)

print(combined.shape)
combined['city'].value_counts()

(14612, 40)


city
BASEL        3653
OSLO         3653
PERPIGNAN    3653
DE_BILT      3653
Name: count, dtype: int64

## Missingness in the combined table

Not every city has every column (Perpignan has no `cloud_cover`/`sunshine`;
Basel has no `wind_speed`/`wind_gust`). These show up as NaN here — that's
expected, not a bug. A handful of NaNs also appear in the lag columns
(exactly `lag × 4 cities` per lag, e.g. 4 for lag1, 8 for lag2, 12 for
lag3) — the first `lag` days of *each* city's series have no prior day to
look back to.

**Downstream note (found and fixed while reviewing the codebase):**
`PhysicalThresholdBaseline` in `baselines.py` used to crash on exactly this
kind of partially-NaN multi-city table — it now automatically falls back
to only the columns that are fully populated across whatever data it's
given, rather than raising.

In [4]:
missing = combined.isna().sum()
missing[missing > 0].sort_values(ascending=False)

wind_gust            7306
sunshine             3653
cloud_cover          3653
wind_speed           3653
temp_min_lag3          12
humidity_pct_lag3      12
temp_min_lag2           8
humidity_pct_lag2       8
temp_min_lag1           4
humidity_pct_lag1       4
temp_min_roc1           4
dtype: int64

## Train / validation / test split

Date-based, not random — this is a time series, and a random split would
leak neighboring days between train and test. Same three date cutoffs
applied to every city, using the shared 2000-2010 calendar all four
stations have.

In [5]:
TRAIN_CUTOFF = pd.Timestamp('2007-01-01')
VAL_CUTOFF = pd.Timestamp('2008-07-01')

train = combined[combined['DATE'] < TRAIN_CUTOFF].copy()
val = combined[(combined['DATE'] >= TRAIN_CUTOFF) & (combined['DATE'] < VAL_CUTOFF)].copy()
test = combined[combined['DATE'] >= VAL_CUTOFF].copy()

print(f"Train: {len(train)} rows ({train['DATE'].min().date()} to {train['DATE'].max().date()})")
print(f"Val:   {len(val)} rows ({val['DATE'].min().date()} to {val['DATE'].max().date()})")
print(f"Test:  {len(test)} rows ({test['DATE'].min().date()} to {test['DATE'].max().date()})")

Train: 10228 rows (2000-01-01 to 2006-12-31)
Val:   2188 rows (2007-01-01 to 2008-06-30)
Test:  2196 rows (2008-07-01 to 2009-12-31)


## Sanity check — do these features actually help?

Not the real Phase 3 model comparison (that's `05_model_training.ipynb`'s
job, with RF/XGBoost/LightGBM/CatBoost) — just a quick check with plain
linear regression on the common features every city has (excluding wind,
since Basel lacks it), to confirm the feature engineering was worth doing
before building the real models on top of it.

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

CORE_FEATURES = [
    'temp_min', 'humidity_pct', 'dew_point_depression_c',
    'temp_min_lag1', 'temp_min_lag2', 'temp_min_lag3',
    'temp_min_roll3_mean', 'temp_min_roll7_mean', 'temp_min_roc1',
    'doy_sin', 'doy_cos',
]

train_c = train.dropna(subset=CORE_FEATURES + ['target_min_temp'])
test_c = test.dropna(subset=CORE_FEATURES + ['target_min_temp'])

persistence_mae = mean_absolute_error(test_c['target_min_temp'], test_c['temp_min'])

lr = LinearRegression().fit(train_c[CORE_FEATURES], train_c['target_min_temp'])
engineered_mae = mean_absolute_error(test_c['target_min_temp'], lr.predict(test_c[CORE_FEATURES]))

print(f"Persistence MAE:          {persistence_mae:.3f}C")
print(f"Engineered features MAE:  {engineered_mae:.3f}C  (plain linear regression)")

Persistence MAE:          2.107C
Engineered features MAE:  1.992C  (plain linear regression)


Even a plain linear regression over the engineered features beats
persistence outright — a good sign heading into Phase 3, where tree
ensembles should do meaningfully better than this linear baseline.

In [7]:
os.makedirs('../data/processed', exist_ok=True)

train.to_csv('../data/processed/train.csv', index=False)
val.to_csv('../data/processed/val.csv', index=False)
test.to_csv('../data/processed/test.csv', index=False)

print("Saved train/val/test to data/processed/")

Saved train/val/test to data/processed/
